<a href="https://colab.research.google.com/github/adhishmodi/customer-churn-prediction/blob/main/notebooks/06_class_imbalance_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 5 — Phase 2A: Class-Imbalance Experiments

Compare balanced and unweighted versions of the existing Logistic Regression, Decision Tree, Random Forest and XGBoost configurations. A Dummy prior classifier is included as a prevalence-only reference.

**Protocol:** fixed hyperparameters, identical repeated stratified CV folds, fold-local preprocessing and training-only evaluation. The held-out test set is not used. The default prediction threshold remains 0.50; calibration and threshold selection are reserved for Phase 3.

## 1. Environment and dataset setup

Run this notebook from the repository root or in Google Colab. The raw IBM Telco CSV is not committed to Git; upload the original dataset when prompted.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/adhishmodi/customer-churn-prediction.git"
EXPECTED_CSV = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
current_dir = Path.cwd()
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "src").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path("/content/customer-churn-prediction")
    if not (PROJECT_ROOT / "src").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_PATH = PROJECT_ROOT / "data" / "raw" / EXPECTED_CSV
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Expected dataset: {DATA_PATH}")

In [ ]:
if not DATA_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Dataset not found at {DATA_PATH}. Place the original CSV there."
        ) from exc
    print(f"Upload the original file named {EXPECTED_CSV}")
    uploaded = files.upload()
    csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
    if EXPECTED_CSV in uploaded:
        selected_name = EXPECTED_CSV
    elif len(csv_names) == 1:
        selected_name = csv_names[0]
    else:
        raise ValueError(
            f"Upload exactly one CSV, preferably named {EXPECTED_CSV}. Found: {csv_names}"
        )
    DATA_PATH.write_bytes(uploaded[selected_name])
assert DATA_PATH.exists(), "Dataset setup did not create the expected CSV file."

## 2. Reconstruct the training partition

The original stratified 80/20 split and seed 42 are reconstructed for reproducibility. The held-out rows are immediately discarded from this experiment and never passed to model comparison, feature selection or tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.preprocess import load_and_clean_data, prepare_features_target, split_data
from src.train import get_imbalance_experiment_models
from src.experiments import compare_models_repeated_cv

cleaned_df = load_and_clean_data(DATA_PATH)
X, y = prepare_features_target(cleaned_df)
X_train, held_out_features, y_train, held_out_labels = split_data(X, y)
del held_out_features, held_out_labels
# Drop the full pre-split frame too; subsequent work uses training rows only.
del X, y, cleaned_df

print(f"Training rows: {len(X_train):,}")
print(f"Training churn prevalence: {y_train.mean():.2%}")
print(f"Training class counts: {y_train.value_counts().sort_index().to_dict()}")
print("Held-out test set used: False")

## 3. Experiment design

The balanced and unweighted members of each pair share the same non-weighting hyperparameters. Scikit-learn class weights are recalculated during each estimator fit. XGBoost's positive-class ratio is computed from that fit's labels; its unweighted variant uses `scale_pos_weight=1.0`.

In [ ]:
candidate_models = get_imbalance_experiment_models()
print("Candidates:")
for candidate_name in candidate_models:
    print(f"- {candidate_name}")

N_SPLITS = 5
N_REPEATS = 3
RANDOM_STATE = 42
print(f"Validation: {N_SPLITS} folds × {N_REPEATS} repeats")
print("Selection metric: ROC-AUC; threshold: 0.50")

## 4. Run repeated stratified cross-validation

Every candidate receives the same 15 train/validation splits. All learned preprocessing remains inside the pipeline and is fitted separately within each training fold. This experiment accepts only `X_train` and `y_train`.

In [ ]:
imbalance_results = compare_models_repeated_cv(
    X_train,
    y_train,
    models=candidate_models,
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)
assert imbalance_results["configuration"]["test_set_used"] is False
assert len(imbalance_results["fold_scores"]) == (
    len(candidate_models) * N_SPLITS * N_REPEATS
)
print(imbalance_results["configuration"])
print("Repeated-CV experiment completed.")

## 5. Compare validation results

Use ROC-AUC and Average Precision to compare ranking quality, then inspect precision/recall/F1/accuracy and fit time. Fold standard deviations describe observed variability; they are not confidence intervals or significance tests.

In [ ]:
summary = imbalance_results["summary"].copy()
display_columns = [
    "model", "fold_count", "roc_auc_mean", "roc_auc_std",
    "average_precision_mean", "average_precision_std",
    "precision_mean", "recall_mean", "f1_mean", "accuracy_mean", "fit_time_mean",
]
display(summary[display_columns].sort_values("roc_auc_mean", ascending=False).round(4))

summary["model_family"] = summary["model"].str.replace(
    r" \((balanced|unweighted)\)$", "", regex=True
)
summary["weighting"] = summary["model"].str.extract(
    r"\((balanced|unweighted)\)$", expand=False
).fillna("reference")

paired_rows = []
for family in ("Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"):
    family_rows = summary[summary["model_family"] == family].set_index("weighting")
    row = {"model_family": family}
    for metric in ("roc_auc", "average_precision", "precision", "recall", "f1", "accuracy"):
        row[f"{metric}_balanced_minus_unweighted"] = float(
            family_rows.loc["balanced", f"{metric}_mean"]
            - family_rows.loc["unweighted", f"{metric}_mean"]
        )
    paired_rows.append(row)
paired_differences = pd.DataFrame(paired_rows)
display(paired_differences.round(4))

In [ ]:
plot_data = summary.set_index("model")[["roc_auc_mean", "average_precision_mean"]]
axis = plot_data.plot(kind="bar", figsize=(12, 5))
axis.set_title("Repeated-CV ranking metrics by class-weight configuration")
axis.set_ylabel("Mean score across folds")
axis.set_xlabel("Candidate configuration")
axis.tick_params(axis="x", rotation=35)
axis.legend(["ROC-AUC", "Average Precision"])
plt.tight_layout()
plt.show()

## 6. Selection stability and interpretation

The repeat winner is the candidate with the highest mean ROC-AUC within each repeat. Winner frequency is descriptive stability evidence, not proof of statistical superiority. A weighted model may increase recall while reducing precision; decide based on the full metric trade-off and the eventual retention use case.

In [ ]:
display(imbalance_results["repeat_winners"])
display(imbalance_results["selection_stability"])

print("Interpretation checklist:")
print("1. Did class weighting improve ROC-AUC or Average Precision consistently?")
print("2. Did it increase recall at the cost of precision?")
print("3. Were changes consistent across repeated folds?")
print("4. Is any gain large enough to justify the additional false positives?")
print("5. Do not select a probability threshold here; that is Phase 3.")

## 7. Save training-only experiment outputs

The generated CSV files are written to the ignored `reports/` directory. They contain only training-CV results and do not replace the frozen Milestone 3–4 baseline.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "phase_2a_imbalance_fold_scores.csv": imbalance_results["fold_scores"],
    "phase_2a_imbalance_summary.csv": summary,
    "phase_2a_imbalance_repeat_winners.csv": imbalance_results["repeat_winners"],
    "phase_2a_imbalance_selection_stability.csv": imbalance_results["selection_stability"],
    "phase_2a_imbalance_paired_differences.csv": paired_differences,
}
for filename, frame in outputs.items():
    frame.to_csv(REPORTS_DIR / filename, index=False)
    print(f"Saved: {REPORTS_DIR / filename}")

## 8. Final protocol checks

Before interpreting the results, confirm the candidate count, number of repeated folds, finite scores and explicit no-test-use flag. Do not copy any metrics into the project report until this notebook has been executed against the original CSV and the outputs have been reviewed.

In [ ]:
assert len(candidate_models) == 9
assert len(imbalance_results["fold_scores"]) == 9 * N_SPLITS * N_REPEATS
metric_columns = [
    "roc_auc", "average_precision", "precision", "recall", "f1", "accuracy"
]
assert np.isfinite(
    imbalance_results["fold_scores"][metric_columns].to_numpy()
).all()
assert imbalance_results["configuration"]["test_set_used"] is False
assert set(paired_differences["model_family"]) == {
    "Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"
}
print("All Phase 2A protocol assertions passed.")
print("Training-only CV rows evaluated:", len(imbalance_results["fold_scores"]))
print("Held-out test set used: False")